# MSG91 OTP delivery + verification demo

This notebook exercises the **real** production code (`app/services/otp.py`, `app/config.py`) end to end
against your own phone number and a real MSG91 account - it sends you an actual SMS and then checks it
against the same local verification logic the API uses (`otp.matches`), rather than calling MSG91's own
Verify API (see the "Must-fix blockers" discussion on why delivery and verification are kept separate).

**Before running:**
- You need a MSG91 account with a Send-OTP template already approved (contains the `##OTP##` variable)
  and its `template_id`, plus your account's `authkey`.
- This notebook asks for your authkey with a hidden prompt (`getpass`) each time you run it. It is **never**
  written to this file or printed anywhere - if you save this notebook with outputs, nothing secret ends
  up in it. Don't hardcode the authkey into a cell.
- It sends one real SMS (and consumes one MSG91 OTP credit) each time you run the "send" cell.
- Run this from the `notebooks/` folder of `backend-api` (or anywhere inside the repo) so the `app` package
  import below resolves.


In [1]:
import sys
from pathlib import Path

# Walk up until we find the `app` package (works whether this notebook lives in notebooks/ or the repo root).
here = Path.cwd()
repo_root = next(p for p in [here, *here.parents] if (p / "app" / "config.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Using backend-api repo at: {repo_root}")

Using backend-api repo at: c:\HomeSchoolingApp\HomeSchoolingApp-Repos\backend-api


## 1. Configure - enter your MSG91 credentials and phone number

None of these values are written to disk by this notebook.

In [4]:
import getpass
import os

os.environ["APP_ENV"] = "dev"
os.environ["OTP_PROVIDER"] = "msg91"
os.environ["MSG91_AUTH_KEY"] = getpass.getpass("MSG91 auth key (hidden input): ")
os.environ["MSG91_TEMPLATE_ID"] = input("MSG91 template id: ").strip()
os.environ["OTP_TTL_MINUTES"] = "10"

phone = input("Your mobile number, with country code (e.g. +919876543210): ").strip()

print("Configured. (Nothing above is echoed or saved - the authkey and phone number only live in this kernel's memory.)")

Configured. (Nothing above is echoed or saved - the authkey and phone number only live in this kernel's memory.)


## 2. Load settings and confirm they picked up what you just entered

In [3]:
from app.config import get_settings, reset_settings_cache
from app.services import otp

reset_settings_cache()  # safe to re-run this notebook in the same kernel without restarting it
s = get_settings()

assert s.otp_provider == "msg91", f"expected msg91, got {s.otp_provider!r}"
assert s.msg91_auth_key.get_secret_value(), "MSG91 auth key did not get set"
assert s.msg91_template_id, "MSG91 template id did not get set"

print(f"otp_provider={s.otp_provider}  template_id={s.msg91_template_id}  otp_ttl_minutes={s.otp_ttl_minutes}")

AssertionError: MSG91 auth key did not get set

## 3. Send a real OTP

This calls `otp.deliver()` - the exact function `POST /v1/families/{id}/guardian-verification` calls in
`app/routers/families.py`. It generates a random 6-digit code, stores only its digest (never the code
itself) exactly like the real endpoint does, and sends the code to MSG91 for SMS delivery.

The code is deliberately **not printed** below - that would defeat the point of testing real delivery. You
should receive it by SMS within a few seconds.

In [ ]:
import uuid

verification_id = uuid.uuid4()
code_sent = otp.new_code()
stored_hash = otp.digest(verification_id, code_sent)

otp.deliver(phone, code_sent)

print(f"Sent. Check the SMS on your phone ending ****{phone[-4:]}.")
print("(The code itself isn't shown here - enter what you actually received in the next step.)")

## 4. Verify - enter the code you received by SMS

Re-run this cell as many times as you like; it doesn't send a new OTP.

In [ ]:
entered_code = input("Enter the OTP you received via SMS: ").strip()
is_valid = otp.matches(verification_id, entered_code, stored_hash)

print("\u2705 Verified - matches the code that was actually sent." if is_valid
      else "\u274c Did not match - re-check the SMS and try this cell again (don't re-run the send cell above).")

## 5. Sanity check - a wrong code must be rejected

Confirms `otp.matches` isn't accidentally accepting everything.

In [ ]:
# A deliberately wrong guess (never the real code_sent, whatever that happened to be) must fail.
wrong_guess = "000001" if code_sent != "000001" else "000002"
assert otp.matches(verification_id, wrong_guess, stored_hash) is False

print("Negative-path check passed: an incorrect code is correctly rejected.")